# 01 — Récupération des données NHANES

NHANES est l'enquête de santé du CDC : chaque participant passe un entretien à domicile
puis un examen médical (prise de sang, mesures, tension). C'est une des rares sources
publiques qui combine **alimentation mesurée** (rappels 24 h) et **biomarqueurs**, ce qui
tombe bien pour la question que je veux traiter.

Je prends trois cycles de deux ans : 2013-2014, 2015-2016, 2017-2018. Remonter plus loin
obligerait à gérer des changements de méthode plus lourds, et trois cycles donnent déjà
assez de monde.

Les fichiers sont au format SAS `.XPT`, un par composant et par cycle. `pandas.read_sas`
les lit directement, pas besoin de SAS. Tout le téléchargement/cache est dans
`src/data_loader.py`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60)

In [2]:
from src import data_loader
from src.config import COMPONENTS, CYCLES

for code, desc in COMPONENTS.items():
    print(f"{code:8s} {desc}")

DEMO     âge, sexe, origine, revenu, éducation, pondérations
BMX      mesures anthropométriques (tour de taille, IMC)
BPX      tension artérielle (plusieurs mesures)
GHB      hémoglobine glyquée (HbA1c)
GLU      glycémie à jeun (sous-échantillon)
HDL      HDL-cholestérol
TRIGLY   triglycérides (sous-échantillon à jeun)
TCHOL    cholestérol total
DR1TOT   rappel alimentaire 24 h, jour 1
DR2TOT   rappel alimentaire 24 h, jour 2
PAQ      activité physique
SMQ      tabac
ALQ      alcool
SLQ      sommeil
DIQ      diabète déclaré / traitement
BPQ      HTA et cholestérol déclarés / traitement
MCQ      antécédents cardiovasculaires déclarés


In [3]:
# Télécharge (ou relit depuis data/raw/) chaque composant, empilé sur les 3 cycles.
tables = data_loader.load_all(list(COMPONENTS), list(CYCLES))

resume = pd.DataFrame({
    code: {"lignes": len(df), "colonnes": df.shape[1],
           **df.groupby("cycle").size().to_dict()}
    for code, df in tables.items()
}).T
resume

,lignes,colonnes,2013-2014,2015-2016,2017-2018
DEMO,29400,52,10175,9971,9254
BMX,28061,29,9813,9544,8704
BPX,28061,24,9813,9544,8704
GHB,20124,3,6979,6744,6401
GLU,9556,7,3329,3191,3036
HDL,23747,4,8291,8021,7435
TRIGLY,9556,11,3329,3191,3036
TCHOL,23747,4,8291,8021,7435
DR1TOT,28061,172,9813,9544,8704
DR2TOT,28061,89,9813,9544,8704


Deux choses à retenir de ce tableau :

- **GLU et TRIGLY n'ont que ~9 500 lignes** contre ~28 000 ailleurs. La glycémie et les
  triglycérides à jeun ne sont mesurés que sur le sous-échantillon convoqué le matin à
  jeun. Comme ce sont deux des cinq critères du syndrome métabolique, ce sous-échantillon
  sera ma population d'analyse. C'est la contrainte qui fixe la taille finale (~5 500).
- **PAQ 2017-2018 est plus court** : le questionnaire activité physique a été raccourci
  cette année-là. Les variables que j'utilise (activité de loisir, sédentarité) restent
  présentes.

Le cache `data/raw/` fait ~130 Mo. Rien n'est écrit dans `data/processed/` à cette étape :
le notebook 02 repart de `load_all()`, qui est instantané une fois le cache constitué.